<a href="https://colab.research.google.com/github/StellaIbeh/ML_Chatbot/blob/main/stella_medical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate

# Import Requirements

In [ ]:
import pandas as pd
import evaluate
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

# Load Dataset

In [ ]:
# Load dataset (example, adjust path as needed)
df = pd.read_csv("/content/domain_specific_chatbot_data (1).csv")

# Display a sample
df.head()

,query,response,intent,domain
0,What are the side effects of the COVID-19 vacc...,Common side effects of the COVID-19 vaccine in...,side effects inquiry,healthcare
1,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
2,What should I do if I miss a dose of my medica...,"If you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
3,How can I check my account balance?,You can check your balance by logging into you...,balance inquiry,finance
4,What is the interest rate for a personal loan?,The current interest rate for personal loans i...,loan inquiry,finance


# Data Preprocessing

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_df.shape, val_df.shape

((2400, 4), (600, 4))

In [ ]:
train_data = train_df.reset_index(drop=True)
validation_data = val_df.reset_index(drop=True)

validation_data

,query,response,intent,domain
0,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
1,What are the side effects of the COVID-19 vacc...,Common side effects of the COVID-19 vaccine in...,side effects inquiry,healthcare
2,How do I update my contact details on my account?,"To update your contact details, log into your ...",contact update,finance
3,How can I schedule an appointment with my doctor?,You can schedule an appointment by calling our...,appointment booking,healthcare
4,"I lost my credit card, what should I do?",Please contact our customer service immediatel...,lost card reporting,finance
...,...,...,...,...
595,What is the interest rate for a personal loan?,The current interest rate for personal loans i...,loan inquiry,finance
596,How do I update my contact details on my account?,"To update your contact details, log into your ...",contact update,finance
597,How do I apply for a student loan?,You can apply for a student loan by visiting o...,student loan application,finance
598,What are the symptoms of flu?,"Flu symptoms include fever, cough, sore throat...",flu symptoms inquiry,healthcare


# Text Cleaning

In [ ]:
# Clean the text by removing unwanted characters
import re

def clean_text(text):
    text = re.sub(r'\r\n', ' ', text)  # Remove carriage returns and line breaks
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'<.*?>', '', text)  # Remove any XML tags
    text = text.strip().lower()  # Strip and convert to lower case
    return text

# Apply cleaning to dialogue and summary columns
train_data['query'] = train_data['query'].apply(clean_text)
train_data['response'] = train_data['response'].apply(clean_text)

validation_data['query'] = validation_data['query'].apply(clean_text)
validation_data['response'] = validation_data['response'].apply(clean_text)


# Display a sample after cleaning
train_data

,query,response,intent,domain
0,what should i do if i miss a dose of my medica...,"if you miss a dose, take it as soon as you rem...",medication inquiry,healthcare
1,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
2,what are the symptoms of flu?,"flu symptoms include fever, cough, sore throat...",flu symptoms inquiry,healthcare
3,how do i update my contact details on my account?,"to update your contact details, log into your ...",contact update,finance
4,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
...,...,...,...,...
2395,can i make changes to my loan repayment schedule?,changes to your loan repayment schedule can be...,loan repayment adjustment,finance
2396,"i lost my credit card, what should i do?",please contact our customer service immediatel...,lost card reporting,finance
2397,what are the side effects of the covid-19 vacc...,common side effects of the covid-19 vaccine in...,side effects inquiry,healthcare
2398,what is the interest rate for a personal loan?,the current interest rate for personal loans i...,loan inquiry,finance


# Tokenization using Bert

In [ ]:

tokenizer= BertTokenizer.from_pretrained("bert-base-uncased")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
# Preprocessing function for tokenization
def preprocess_function(examples):
    # Tokenize the dialogue and summary
    inputs = tokenizer(examples["query"], padding="max_length", truncation=True, max_length=250)
    targets = tokenizer(examples["response"], padding="max_length", truncation=True, max_length=250)
    inputs["labels"] = targets["input_ids"]
    return inputs

# Apply the preprocessing
train_dataset = train_data.apply(preprocess_function, axis=1)
val_dataset = validation_data.apply(preprocess_function, axis=1)


In [ ]:
train_data['response'][0]

"if you miss a dose, take it as soon as you remember unless it's almost time for your next dose. if you’re unsure, contact your healthcare provider."

In [ ]:
train_dataset[0]


{'input_ids': [125, 225, 3, 23, 103, 3, 99, 3, 23, 3041, 3, 9, 6742, 13, 82, 7757, 58, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Fine Tuning Model and Model Training

In [ ]:
# Model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # output directory for checkpoints
    num_train_epochs=6,              # number of training epochs
    per_device_train_batch_size=8,   # batch size per device during training
    per_device_eval_batch_size=8,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir="./logs",            # directory for storing logs
    logging_steps=50,                # how often to log training info
    save_steps=500,                  # how often to save a model checkpoint
    eval_steps=50,                   # how often to run evaluation
    evaluation_strategy="epoch",     # Ensure evaluation happens every `epoch`
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train the model
trainer.train()

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.275000,0.192064
2,0.026800,0.006303
3,0.007700,0.000730
4,0.003800,0.000213
5,0.002900,0.000119
6,0.002600,0.000102


TrainOutput(global_step=1800, training_loss=0.8489711766276095, metrics={'train_runtime': 305.9357, 'train_samples_per_second': 47.069, 'train_steps_per_second': 5.884, 'total_flos': 951622041600000.0, 'train_loss': 0.8489711766276095, 'epoch': 6.0})

In [ ]:
model.save_pretrained("./chatbot_model")
tokenizer.save_pretrained("./chatbot_model")


model = T5ForConditionalGeneration.from_pretrained("./chatbot_model")
tokenizer = T5Tokenizer.from_pretrained("./chatbot_model")

In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 52.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993222 sha256=07cf5301df4d6ac26da1aac73201fdfd5b840874319528d6943c59886af04c43
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect


In [ ]:
from langdetect import detect

# Example input text
input_text = "Wie kann ich einen Termin mit meinem Arzt vereinbaren?"

# Detect the language of the input
language = detect(input_text)
print(f"Detected language: {language}")


In [ ]:
from transformers import MBartForConditionalGeneration, MBartTokenizer
from langdetect import detect

# Load the mBART tokenizer and model (this is multilingual)
tokenizer = MBartTokenizer.from_pretrained("facebook/mbart-large-50-one-to-many-mmt")
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-one-to-many-mmt")

# Function to generate responses in the detected language
def generate_response(input_text):
    # Detect the language of the input
    language = detect(input_text)

    # Set the target language code (for example, 'en' for English, 'de' for German)
    if language == 'en':
        target_lang = 'en_XX'  # English code for mBART
    elif language == 'de':
        target_lang = 'de_DE'  # German code for mBART
    else:
        target_lang = 'en_XX'  # Default to English

    # Tokenize the input text
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

    # Generate the output
    generated_ids = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[target_lang]  # Set the language for the response
    )

    # Decode the generated text
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return output_text

# Example input in German
input_text = "Wie kann ich einen Termin mit meinem Arzt vereinbaren?"
response = generate_response(input_text)
print(f"Response: {response}")


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Response: Wie kann ich einen Termin mit meinem Arzt vereinbaren?


In [ ]:
device = model.device


def chatbot(query):
    query = clean_text(query)
    input_ids = tokenizer(query,return_tensors="pt",max_length=250,truncation=True)

    inputs = {key: value.to(device) for key, value in input_ids.items()}

    outputs = model.generate(
        input_ids["input_ids"],
        max_length=250,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    response = chatbot(user_input)
    print("Chatbot:", response)

Chatbot: ដើម្បី ចូល ទៅ កាន់ ប្រព័ន្ធ យ៉ាង ដូច ម្ដេច?
Chatbot: कहाँ सेटिंग विकल्प खोजने के लिए?
Chatbot: विकल्प खोजने के लिए?
Chatbot: どうやって医師に訴えるの?


In [ ]:
# calculate perplexity
import math
from torch.utils.data import DataLoader


In [ ]:
# Evaluate on Test Set
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.00010200436372542754, 'eval_runtime': 3.5856, 'eval_samples_per_second': 167.337, 'eval_steps_per_second': 20.917, 'epoch': 6.0}


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('t5-small')  # Use your model name
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')  # Use your model name

def generate_predictions(val_dataset):
    predictions = []

    for example in val_dataset:
        input_ids = torch.tensor(example['input_ids']).unsqueeze(0)  # Add batch dimension
        attention_mask = torch.tensor(example['attention_mask']).unsqueeze(0)

        # Generate predictions using the model
        output = model.generate(input_ids, attention_mask=attention_mask)

        # Decode the output (predicted tokens) back into text
        prediction_text = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(prediction_text)

    return predictions

# Generate predictions
predictions = generate_predictions(val_dataset)

# Print some predictions to check
print(predictions[:5])


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

['Wie kann ich ein Termin Termin mit meinem Arzt vereinbaren?', 'Welche Nebenwirkungwirkung hat der Covid-19 vaccine?', 'Wie aktualisiere ich meine Kontaktdaten auf meinem Account?', 'Wie kann ich ein Termin Termin mit meinem Arzt vereinbaren?', 'Ich habe verloren meine Kreditkarte, was sollte ich tun?']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('t5-small')  # Change to your model name
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')  # Change to your model name

# Load BLEU metric
bleu = evaluate.load("bleu")

# Function to generate predictions
def generate_predictions(dataset):
    predictions = []
    for example in dataset:
        # Assuming the dataset has 'input_text' as the raw input
        input_text = example["input_text"]  # Adjust if your dataset has a different key for input

        # Tokenize the input text
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

        # Generate the output
        outputs = model.generate(**inputs)

        # Decode the output tokens to text
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(prediction)

    return predictions

# Generate predictions
predictions = generate_predictions(val_dataset)

# Prepare references (assuming 'labels' are tokenized)
references = [[tokenizer.decode(example["labels"], skip_special_tokens=True)] for example in val_dataset]

# Compute BLEU score
bleu_score = bleu.compute(predictions=predictions, references=references)
print(f"BLEU Score: {bleu_score}")

# Load F1 metric
f1_metric = evaluate.load("f1")

# Compute F1 score
f1_score = f1_metric.compute(predictions=predictions, references=references)

print(f"F1 Score: {f1_score}")




KeyError: 'input_text'